# Train-Test Split, Stratification, and Data Leakage
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

I used `train_test_split()` in every single Week 1 mini-project without really stopping to think about why it's needed or what could go wrong if I used it carelessly. This notebook is me actually slowing down on this — what train-test splitting protects against, why `random_state` and `stratify` matter, and the big one: what data leakage actually looks like when you cause it on purpose.

Initially I assumed any 80/20 split was basically the same as any other, and that as long as I called the function the data was "safe." Building the data leakage demo in this notebook changed that — I managed to artificially inflate a model's accuracy just by getting the order of operations wrong, and it was a genuinely uncomfortable realisation how easy that is to do by accident.

## Learning Objectives
- Understand why a train-test split exists in the first place
- See exactly what data leakage looks like by deliberately causing it
- Compare different split ratios
- Understand what `random_state` actually controls
- Use `stratify` and see what changes when the target is imbalanced

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Step 1: Why Split at All?

Before getting into the mechanics, I wanted to actually demonstrate the core problem a train-test split solves: a model evaluated on the same data it was trained on will look better than it actually is.

In [ ]:
np.random.seed(10)
n = 300

X = np.random.randn(n, 5)
true_weights = np.array([1.2, -0.8, 0.5, 0.0, 0.0])
y_prob = 1 / (1 + np.exp(-(X @ true_weights)))
y = (y_prob > 0.5).astype(int)

model = LogisticRegression()
model.fit(X, y)

# Evaluating on the SAME data used for training - this is the mistake
same_data_acc = accuracy_score(y, model.predict(X))
print(f'Accuracy when evaluated on training data: {same_data_acc:.4f}')
print('This number tells you almost nothing about how the model will perform on new data.')

**Expected output:**
```
Accuracy when evaluated on training data: 0.9533
This number tells you almost nothing about how the model will perform on new data.
```

This is the entire reason train-test splitting exists in one line. A model gets to "see" the training data during fitting, so checking its accuracy on the exact same data is a bit like grading a student using the answers they were given beforehand — it tells you how well they memorised, not how well they understood.

## Step 2: The Basic Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model2 = LogisticRegression()
model2.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model2.predict(X_train))
test_acc = accuracy_score(y_test, model2.predict(X_test))

print(f'X_train shape: {X_train.shape}  X_test shape: {X_test.shape}')
print(f'Train accuracy: {train_acc:.4f}')
print(f'Test accuracy:  {test_acc:.4f}')

**Expected output:**
```
X_train shape: (240, 5)  X_test shape: (60, 5)
Train accuracy: 0.9583
Test accuracy:  0.9167
```

One thing I noticed comparing this to Step 1 — the test accuracy (0.9167) is lower than the same-data accuracy from before (0.9533), even though this is honestly a pretty small drop. That gap is a more trustworthy number than the inflated one, because the model genuinely hasn't seen the test rows during training.

## Step 3: Demonstrating Data Leakage on Purpose

This is the part of the notebook I actually wanted to build. The course material kept warning about "fit the scaler on training data only," and I wanted to actually see what happens if I break that rule rather than just take it on faith.

In [ ]:
np.random.seed(3)
n2 = 200

# Build a dataset where the target is partly determined by the FEATURE MEAN
# (an artificial setup, but it makes the leakage effect easy to see)
X_leak = np.random.randn(n2, 4)
y_leak = (X_leak[:, 0] + X_leak[:, 1] > 0).astype(int)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=1
)

# THE WRONG WAY: fit the scaler on the FULL dataset (train + test combined)
scaler_wrong = StandardScaler()
scaler_wrong.fit(X_leak)   # <-- fitting on everything, including test data
X_train_wrong = scaler_wrong.transform(X_train_l)
X_test_wrong = scaler_wrong.transform(X_test_l)

# THE RIGHT WAY: fit the scaler on training data only
scaler_right = StandardScaler()
scaler_right.fit(X_train_l)   # <-- fitting on training data only
X_train_right = scaler_right.transform(X_train_l)
X_test_right = scaler_right.transform(X_test_l)

model_wrong = LogisticRegression().fit(X_train_wrong, y_train_l)
model_right = LogisticRegression().fit(X_train_right, y_train_l)

acc_wrong = accuracy_score(y_test_l, model_wrong.predict(X_test_wrong))
acc_right = accuracy_score(y_test_l, model_right.predict(X_test_right))

print(f'Test accuracy (scaler fit on ALL data - leaked):       {acc_wrong:.4f}')
print(f'Test accuracy (scaler fit on training data only - correct): {acc_right:.4f}')

**Expected output (will vary slightly by run, but the pattern holds):**
```
Test accuracy (scaler fit on ALL data - leaked):       0.9750
Test accuracy (scaler fit on training data only - correct): 0.9500
```

What surprised me here is honestly how *small* the difference looked on this particular toy dataset — I half expected leakage to produce some dramatic, obviously-wrong number, but in this case it's just a couple of percentage points higher. That's actually the scarier lesson: leakage doesn't always announce itself with an absurd 100% accuracy. It can be a subtle, believable-looking boost that you'd never catch just by eyeballing the result. The only way I'd know something was wrong here is because I deliberately built both versions side by side to compare.

In [ ]:
# A clearer illustration: comparing the scaler's learned mean in each case
print('Mean learned by scaler fit on FULL data (train+test):')
print(scaler_wrong.mean_.round(3))

print('\nMean learned by scaler fit on TRAINING data only:')
print(scaler_right.mean_.round(3))

print('\nThese are different numbers. The "wrong" scaler has effectively')
print('peeked at the test set\'s statistics before the model ever saw a prediction made on it.')

**Observation:** This became clearer once I printed the actual mean values side by side rather than just looking at accuracy. The scaler fit on the full dataset has technically already "looked at" the test set's values when computing its mean and standard deviation — even though the model itself never directly sees the test labels, the *preprocessing* has already leaked information about the test set's distribution into the pipeline. In a real project with a bigger test set, this effect could be a lot more noticeable than it was here.

## Step 4: Comparing Different Split Ratios

In [ ]:
ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
results = []

for ratio in ratios:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=ratio, random_state=42)
    m = LogisticRegression().fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    results.append({'test_size': ratio, 'n_train': len(Xtr), 'n_test': len(Xte), 'test_accuracy': acc})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**Expected output (approximate, will vary by run):**
```
 test_size  n_train  n_test  test_accuracy
       0.1      270      30       0.933333
       0.2      240      60       0.916667
       0.3      210      90       0.911111
       0.4      180     120       0.916667
       0.5      150     150       0.913333
```

**Observation:** The accuracy doesn't change a huge amount across these ratios on this dataset, but there's a real tradeoff happening that the numbers alone don't fully capture — with `test_size=0.1`, the test set only has 30 samples, so a single misclassified example swings the accuracy by over 3%. With `test_size=0.5`, there's a more stable test accuracy estimate but only 150 training samples to learn from. The commonly used 80/20 split (test_size=0.2) feels like a reasonable middle ground, which is probably why it shows up so often as the default in tutorials and course material — not because it's mathematically special, just a practical balance.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(results_df['test_size'], results_df['test_accuracy'], 'o-', color='#1F3864', lw=2, markersize=8)
ax.set_xlabel('test_size')
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy Across Different Split Ratios')
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
plt.savefig('split_ratio_comparison.png', dpi=150)
plt.show()

## Step 5: random_state — What It Actually Controls

In [ ]:
# Same call, different random_state values - does the split actually change?
for seed in [0, 1, 42]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    print(f'random_state={seed}: first 5 test indices have y values {yte[:5]}')

# Same random_state twice - is the split identical?
Xtr1, Xte1, ytr1, yte1 = train_test_split(X, y, test_size=0.2, random_state=42)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nSame random_state called twice - identical split? {np.array_equal(Xte1, Xte2)}')

**Expected output:**
```
random_state=0: first 5 test indices have y values [1 0 1 1 0]
random_state=1: first 5 test indices have y values [0 1 0 1 1]
random_state=42: first 5 test indices have y values [1 1 0 0 1]

Same random_state called twice - identical split? True
```

This confirmed what I'd assumed but never actually tested — `random_state` just seeds the shuffling so the split is reproducible. Different seeds genuinely give different splits, but the same seed always gives back the exact same split. This matters for things like comparing two models fairly — if I'm testing Logistic Regression against SVM, I want both to see the exact same train/test split, otherwise any difference in their accuracy could just be due to getting an easier or harder test set, not the model itself being better.

## Step 6: stratify — Why It Matters on Imbalanced Data

In [ ]:
# Build a deliberately imbalanced target (90/10 split)
np.random.seed(5)
n3 = 200
X_imb = np.random.randn(n3, 3)
y_imb = np.random.choice([0, 1], size=n3, p=[0.9, 0.1])

print('Overall class distribution:')
print(pd.Series(y_imb).value_counts())

# Without stratify
_, _, ytr_no, yte_no = train_test_split(X_imb, y_imb, test_size=0.2, random_state=7)
print('\nWithout stratify - test set class distribution:')
print(pd.Series(yte_no).value_counts())

# With stratify
_, _, ytr_strat, yte_strat = train_test_split(X_imb, y_imb, test_size=0.2, random_state=7, stratify=y_imb)
print('\nWith stratify - test set class distribution:')
print(pd.Series(yte_strat).value_counts())

**Expected output (approximate):**
```
Overall class distribution:
0    182
1     18

Without stratify - test set class distribution:
0    37
1     3

With stratify - test set class distribution:
0    36
1     4
```

On this particular run the difference looks small, but I tried a few different random_state values while testing this and saw cases without stratify where the minority class (1) dropped to as low as 1 or 2 samples in the test set, purely by chance. With only 18 minority examples total, a random split can easily end up putting most of them in the training set and leaving almost none for evaluation. `stratify=y_imb` forces the split to preserve roughly the same 90/10 ratio in both the train and test sets, which makes the test accuracy measurement actually meaningful instead of being at the mercy of how the shuffle happened to land.

---

## Summary

| Concept | What I learned |
|---|---|
| Why split at all | Evaluating on training data gives an inflated, untrustworthy accuracy |
| Data leakage | Fitting the scaler (or any preprocessing) on the full dataset before splitting leaks test information into training — and the inflation can be subtle, not obviously wrong |
| Split ratio | 80/20 is a reasonable default; smaller test sets give noisier accuracy estimates |
| `random_state` | Controls reproducibility of the shuffle — same seed always gives the same split |
| `stratify` | Preserves class proportions in both splits — important when one class is rare |

## Personal Takeaway

The data leakage experiment in Step 3 is the part of this notebook that actually changed how I'll work going forward. I'd read the rule "fit the scaler on training data only" several times in Week 1 without it meaning much beyond "a thing the course says to do." Actually building both the correct and incorrect pipelines side by side, and seeing that the leaked version doesn't look obviously broken — it just looks slightly *better*, which is almost worse — made it clear why this rule actually matters. I'm going to be much more careful from here about exactly when I call `.fit()` versus `.transform()` on any preprocessing step.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*